In [1]:
import numpy as np
import scipy.special
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

# --- 1. SETUP EXPERIMENT DATA ---
# Create a synthetic classification dataset for the SVM to practice on
X, y = make_classification(n_samples=500, n_features=10, n_informative=8,
                           n_redundant=2, random_state=42)

# --- 2. CUCKOO SEARCH HYPERPARAMETERS ---
NUM_NESTS = 10         # Total number of candidate hyperparameter sets (nests)
ITERATIONS = 30        # How many generations the cuckoos explore
PA = 0.25              # Probability (pa) that a host discovers a cuckoo egg
ALPHA_STEP = 0.1       # Step size scale factor for Levy flights

# Bounds for SVM parameters: [C, gamma]
LOWER_BOUNDS = np.array([0.1, 0.001])
UPPER_BOUNDS = np.array([100.0, 10.0])

# --- 3. CORE UTILITY FUNCTIONS ---
def objective_function(hyperparameters):
    """
    Evaluates the fitness of an SVM using the parameters [C, gamma].
    Returns the classification error rate (1 - accuracy) to minimize.
    """
    C_val = max(LOWER_BOUNDS[0], min(hyperparameters[0], UPPER_BOUNDS[0]))
    gamma_val = max(LOWER_BOUNDS[1], min(hyperparameters[1], UPPER_BOUNDS[1]))

    clf = SVC(C=C_val, gamma=gamma_val, random_state=42)
    # Perform a fast 3-fold cross validation score check
    scores = cross_val_score(clf, X, y, cv=3, n_jobs=-1)

    # Return error rate (Lower error means a better nest solution)
    return 1.0 - np.mean(scores)

def levy_flight(step_size_alpha, dimensions):
    """
    Generates a step vector derived from a stable Levy distribution step size.
    Uses Mantegna's mathematical algorithm layout.
    """
    beta = 1.5
    sigma = (scipy.special.gamma(1 + beta) * np.sin(np.pi * beta / 2) /
             (scipy.special.gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)

    u = np.random.normal(0, sigma, size=dimensions)
    v = np.random.normal(0, 1, size=dimensions)

    step = u / (np.abs(v)**(1 / beta))
    return step_size_alpha * step

# --- 4. INITIALIZE POPULATION ---
# Spawn initial nests randomly scattered inside parameter limits
nests = np.random.uniform(LOWER_BOUNDS, UPPER_BOUNDS, size=(NUM_NESTS, 2))
fitness = np.array([objective_function(nest) for nest in nests])

# Track global absolute champion parameters
best_idx = np.argmin(fitness)
best_nest = nests[best_idx].copy()
best_fitness = fitness[best_idx]

print("Starting Cuckoo Search for SVM Hyperparameter Optimization...\n")

# --- 5. OPTIMIZATION LOOP ---
for iteration in range(ITERATIONS):

    # --- PHASE A: Generate a New Cuckoo Nest via Levy Flight ---
    random_cuckoo_idx = np.random.randint(0, NUM_NESTS)
    current_cuckoo = nests[random_cuckoo_idx].copy()

    # Calculate leap direction
    step = levy_flight(ALPHA_STEP, dimensions=2)
    new_cuckoo = current_cuckoo + step * (current_cuckoo - best_nest)

    # Keep solution legal within search boundaries
    new_cuckoo = np.clip(new_cuckoo, LOWER_BOUNDS, UPPER_BOUNDS)
    new_fitness = objective_function(new_cuckoo)

    # Pick a random host nest to compete against
    random_host_idx = np.random.randint(0, NUM_NESTS)
    if new_fitness < fitness[random_host_idx]:
        nests[random_host_idx] = new_cuckoo
        fitness[random_host_idx] = new_fitness

    # --- PHASE B: Host Discovers Parasite Eggs (Abandon Bad Nests) ---
    for i in range(NUM_NESTS):
        if np.random.rand() < PA:
            # Generate a completely new replacement nest nearby an existing group
            random_n1 = nests[np.random.randint(0, NUM_NESTS)]
            random_n2 = nests[np.random.randint(0, NUM_NESTS)]
            step_factor = np.random.rand()

            nests[i] = nests[i] + step_factor * (random_n1 - random_n2)
            nests[i] = np.clip(nests[i], LOWER_BOUNDS, UPPER_BOUNDS)
            fitness[i] = objective_function(nests[i])

    # --- PHASE C: Update Global Standings ---
    current_best_idx = np.argmin(fitness)
    if fitness[current_best_idx] < best_fitness:
        best_fitness = fitness[current_best_idx]
        best_nest = nests[current_best_idx].copy()

    print(f"Iteration {iteration + 1:02d}/{ITERATIONS} | Minimum Error: {best_fitness:.4f} (Best Accuracy: {(1.0 - best_fitness)*100:.2f}%)")

# --- 6. OUTPUT FINAL RESULTS ---
print("\n" + "="*50)
print("OPTIMIZATION COMPLETE")
print("="*50)
print(f"Optimized C (Regularization):  {best_nest[0]:.4f}")
print(f"Optimized Gamma (Kernel Width): {best_nest[1]:.4f}")
print(f"Maximum Cross-Validation Acc:   {(1.0 - best_fitness)*100:.2f}%")
print("="*50)


Starting Cuckoo Search for SVM Hyperparameter Optimization...

Iteration 01/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 02/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 03/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 04/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 05/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 06/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 07/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 08/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 09/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 10/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 11/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 12/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 13/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 14/30 | Minimum Error: 0.1240 (Best Accuracy: 87.60%)
Iteration 15/30 | Minimum Error: 0.1240 (